In [1]:
import pandas as pd
import numpy as np

file_path = "../data/processed/student_placement_cleaned.csv"

df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset Shape: (12000, 13)

Columns:
['gender', 'age', 'degree', 'branch', 'cgpa', 'backlogs', 'internships', 'certifications', 'coding_skills', 'communication_skills', 'aptitude_score', 'projects', 'placed']


In [2]:
X = df.drop("placed", axis=1)
y = df["placed"]

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nTarget Distribution:")
print(y.value_counts())

Feature Shape: (12000, 12)
Target Shape: (12000,)

Target Distribution:
placed
0    8348
1    3652
Name: count, dtype: int64


In [3]:
categorical_features = [
    "gender",
    "degree",
    "branch"
]

numerical_features = [
    "age",
    "cgpa",
    "backlogs",
    "internships",
    "certifications",
    "coding_skills",
    "communication_skills",
    "aptitude_score",
    "projects"
]

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

print("\nTotal Features:", len(categorical_features) + len(numerical_features))

Categorical Features:
['gender', 'degree', 'branch']

Numerical Features:
['age', 'cgpa', 'backlogs', 'internships', 'certifications', 'coding_skills', 'communication_skills', 'aptitude_score', 'projects']

Total Features: 12


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)
print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

Training Features: (9600, 12)
Testing Features: (2400, 12)
Training Target: (9600,)
Testing Target: (2400,)

Training Target Distribution:
placed
0    6678
1    2922
Name: count, dtype: int64

Testing Target Distribution:
placed
0    1670
1     730
Name: count, dtype: int64


In [7]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    drop="first",
    sparse_output=False
)

X_train_encoded = encoder.fit_transform(X_train[categorical_features])
X_test_encoded = encoder.transform(X_test[categorical_features])

print("Encoded Training Shape:", X_train_encoded.shape)
print("Encoded Testing Shape:", X_test_encoded.shape)

Encoded Training Shape: (9600, 9)
Encoded Testing Shape: (2400, 9)


In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train[numerical_features])
X_test_scaled = scaler.transform(X_test[numerical_features])

print("Scaled Training Shape:", X_train_scaled.shape)
print("Scaled Testing Shape:", X_test_scaled.shape)

Scaled Training Shape: (9600, 9)
Scaled Testing Shape: (2400, 9)


In [9]:
encoded_feature_names = encoder.get_feature_names_out(categorical_features)

print("Encoded Feature Names:")
print(encoded_feature_names)

Encoded Feature Names:
['gender_Male' 'degree_BE' 'degree_BSc' 'degree_BTech' 'branch_CS'
 'branch_DS' 'branch_Electrical' 'branch_IT' 'branch_Mechanical']


In [10]:
all_feature_names = list(encoded_feature_names) + numerical_features

X_train_processed = np.hstack([
    X_train_encoded,
    X_train_scaled
])

X_test_processed = np.hstack([
    X_test_encoded,
    X_test_scaled
])

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=all_feature_names
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=all_feature_names
)

print("Processed Training Shape:", X_train_processed.shape)
print("Processed Testing Shape:", X_test_processed.shape)

print("\nProcessed Features:")
print(X_train_processed.columns.tolist())

Processed Training Shape: (9600, 18)
Processed Testing Shape: (2400, 18)

Processed Features:
['gender_Male', 'degree_BE', 'degree_BSc', 'degree_BTech', 'branch_CS', 'branch_DS', 'branch_Electrical', 'branch_IT', 'branch_Mechanical', 'age', 'cgpa', 'backlogs', 'internships', 'certifications', 'coding_skills', 'communication_skills', 'aptitude_score', 'projects']


In [11]:
print("Missing Values in Training Data:")
print(X_train_processed.isnull().sum().sum())

print("\nMissing Values in Testing Data:")
print(X_test_processed.isnull().sum().sum())

print("\nInfinite Values in Training Data:")
print(np.isinf(X_train_processed).sum().sum())

print("\nInfinite Values in Testing Data:")
print(np.isinf(X_test_processed).sum().sum())

Missing Values in Training Data:
0

Missing Values in Testing Data:
0

Infinite Values in Training Data:
0

Infinite Values in Testing Data:
0


In [12]:
print("First 5 Processed Training Rows:")
print(X_train_processed.head())

First 5 Processed Training Rows:
   gender_Male  degree_BE  degree_BSc  degree_BTech  branch_CS  branch_DS  \
0          1.0        0.0         0.0           0.0        0.0        1.0   
1          0.0        0.0         0.0           0.0        0.0        1.0   
2          0.0        0.0         0.0           0.0        0.0        0.0   
3          1.0        1.0         0.0           0.0        0.0        1.0   
4          0.0        1.0         0.0           0.0        0.0        0.0   

   branch_Electrical  branch_IT  branch_Mechanical       age      cgpa  \
0                0.0        0.0                0.0  0.714968 -1.112856   
1                0.0        0.0                0.0 -0.691526 -0.629741   
2                0.0        0.0                0.0 -0.691526  0.255970   
3                0.0        0.0                0.0  0.714968 -0.951818   
4                1.0        0.0                0.0 -0.691526  1.689210   

   backlogs  internships  certifications  coding_skills  co

In [13]:
import os

os.makedirs("../models", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

X_train_processed.to_csv(
    "../data/processed/X_train_processed.csv",
    index=False
)

X_test_processed.to_csv(
    "../data/processed/X_test_processed.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

print("PROCESSED DATA SAVED")
print("Training features:", X_train_processed.shape)
print("Testing features:", X_test_processed.shape)

PROCESSED DATA SAVED
Training features: (9600, 18)
Testing features: (2400, 18)


In [14]:
import joblib

joblib.dump(
    encoder,
    "../models/encoder.pkl"
)

joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

print("PREPROCESSING OBJECTS SAVED")
print("Encoder: models/encoder.pkl")
print("Scaler: models/scaler.pkl")

PREPROCESSING OBJECTS SAVED
Encoder: models/encoder.pkl
Scaler: models/scaler.pkl
